In [3]:
import datetime 
from dateutil.relativedelta import relativedelta
import pandas as pd
import numpy as np
import db_dtypes
import pymssql
from shutil import copyfile
from openpyxl import load_workbook
import os
from google.cloud import bigquery
os.environ['GOOGLE_APPLICATION_CREDENTIALS'] = '../../../BQ.json'

DB_info = {'server':'192.168.61.119:7622', 'user':'BAReporting', 'password':'KeHeCReme8he'}

client = bigquery.Client()

In [4]:
# original code
sql = r"""
DECLARE sql STRING;

SET sql = (
  SELECT STRING_AGG(
    FORMAT(""""""
      SELECT
        EXTRACT(MONTH FROM `Time`) AS _month,
        REGEXP_EXTRACT(EventLabel, r'^(.+?\\.tab)') AS _EventLabel,
        DeviceId
      FROM `openrice-production.ORGA.SV_%s`
      WHERE platform IN ('android','ios')
        AND region = 'HK'
        AND (EventLabel LIKE '%%HK.LMS2.21.tab%%'
             OR EventLabel LIKE '%%HK.LMS2.9313.tab%%'
        )
    """""", FORMAT_DATE('%Y%m%d', d)),
    "" UNION ALL ""
  )
  FROM UNNEST(GENERATE_DATE_ARRAY(DATE '2026-09-01', DATE '2026-09-30')) AS d
);

EXECUTE IMMEDIATE FORMAT(""""""
  SELECT
    _month, _EventLabel, 
    count(1) as cnt, 
    COUNT(DISTINCT DeviceId) AS device_cnt
  FROM (%s)
  GROUP BY _month, _EventLabel
"""""", sql);
"""



In [ ]:
# LMS Monthly PV / UU

sql = r"""
DECLARE sql STRING;

SET sql = (
  SELECT STRING_AGG(
    FORMAT('''
      SELECT
        EXTRACT(MONTH FROM `Time`) AS _month,
        REGEXP_EXTRACT(EventLabel, r'^(.+?[.]tab)') AS _EventLabel,
        DeviceId
      FROM `openrice-production.ORGA.SV_%s`
      WHERE platform IN ('android', 'ios')
        AND region = 'HK'
        AND (
          EventLabel LIKE '%%HK.LMS2.21.tab%%'
          OR EventLabel LIKE '%%HK.LMS2.9313.tab%%'
        )
    ''', FORMAT_DATE('%Y%m%d', d)),
    ' UNION ALL '
  )
  FROM UNNEST(
    GENERATE_DATE_ARRAY(DATE '2026-09-01', DATE '2026-09-30')
  ) AS d
);

EXECUTE IMMEDIATE FORMAT('''
  SELECT
    _month,
    _EventLabel,
    COUNT(*) AS cnt,
    COUNT(DISTINCT DeviceId) AS device_cnt
  FROM (%s)
  GROUP BY _month, _EventLabel
  ORDER BY _month, _EventLabel
''', sql);
"""

In [10]:
df_lms_monthly = client.query(sql).to_dataframe()
df_lms_monthly

C:\Users\lenalee\AppData\Roaming\Python\Python313\site-packages\google\cloud\bigquery\table.py:2128: UserWarning: BigQuery Storage module not found, fetch data with the REST endpoint instead.
  warnings.warn(


,_month,_EventLabel,cnt,device_cnt
0,9,HK.LMS2.21.tab,51791,13849
1,9,HK.LMS2.9313.tab,39890,22085


In [ ]:
# No. of Search Landmark (nsl)

sql_1 = r"""
DECLARE sql STRING;

SET sql = (
  SELECT STRING_AGG(
    FORMAT('''
      SELECT
        deviceid,
        LOWER(eventlabel) AS screen,
        LOWER(LAG(eventlabel) OVER(ORDER BY sessionid, time)) AS prev_screen
      FROM `openrice-production.ORGA.SV_%s`
      WHERE (LOWER(eventlabel) LIKE 'hk.lms2.%%.tab.-%%'
        OR LOWER(eventlabel) = 'hk.qcksearch'
        OR LOWER(eventlabel) LIKE 'hk.advsearch')
    ''', FORMAT_DATE('%Y%m%d', d))
  , ' UNION ALL ')
  FROM UNNEST(GENERATE_DATE_ARRAY('2026-09-01','2026-09-30')) d
);

EXECUTE IMMEDIATE
FORMAT(
'''
WITH temp AS (
  %s
)

SELECT
  SUM(CASE WHEN prev_screen = 'hk.qcksearch' THEN 1 END) AS quick_,
  SUM(CASE WHEN prev_screen = 'hk.advsearch' THEN 1 END) AS adv_,
  COUNT(DISTINCT deviceid) AS uu,
  screen
FROM temp
WHERE (screen = 'hk.lms2.21.tab.-990.1'
   OR screen = 'hk.lms2.9313.tab.-990.1')
  AND prev_screen LIKE '%%search%%'
GROUP BY screen
ORDER BY adv_ DESC
''',
sql
);
"""


In [ ]:
df_nsl = client.query(sql_1).to_dataframe()
df_nsl

In [ ]:
# LMS Restaurants Total Impressions / Total clicks by LMS (LMS_rest)

sql_2 = r"""
DECLARE sql STRING;

SET sql = (
  SELECT STRING_AGG(
    FORMAT('''
      SELECT
        EventAction,
        event.item.Value AS value
      FROM `openrice-production.ORGA.PV_%s`,
      UNNEST(EventLabel.list) AS event
      WHERE EventAction IN ('impression.poi', 'or.poi.get-details')
        AND event.item.Param = 'sn'
        AND event.item.Value IN ('HK.LMS2.21.tab.-990.1','HK.LMS2.9313.tab.-990.1')
    ''', FORMAT_DATE('%Y%m%d', d))
  , ' UNION ALL ')
  FROM UNNEST(GENERATE_DATE_ARRAY('2026-09-01','2026-09-30')) d
);

EXECUTE IMMEDIATE
FORMAT(
'''
SELECT
  EventAction,
  value,
  COUNT(1) AS cnt
FROM (
  %s
)
GROUP BY EventAction, value
''',
sql
);
"""


In [ ]:
df_lms_rest = client.query(sql_2).to_dataframe()
df_lms_rest

In [ ]:
# Top 5 Search Restaurant / Top 5 Tenant Viewed (top5)

sql_3 = r"""
DECLARE query STRING;

## Build per-day UNION ALL query
SET query = (
  WITH all_dates AS (
    SELECT
      EXTRACT(YEAR FROM dt)  AS year,
      EXTRACT(MONTH FROM dt) AS month,
      EXTRACT(DAY FROM dt)   AS day
    FROM UNNEST(GENERATE_DATE_ARRAY('2026-09-01', '2026-09-30')) AS dt
  )
  SELECT STRING_AGG(subquery, ' UNION ALL ')
  FROM (
    SELECT FORMAT(
      '''
      SELECT
        SAFE_CAST(event.value.string_value AS INT64) AS poiid
      FROM `openrice-production.analytics_153356786.cluster_events_%04d%02d%02d`,
           UNNEST(event_params) AS event
      WHERE
         user_pseudo_id LIKE '1%%%%'
        AND event_name = 'or_poi_get_details'
        AND event.key = 'POIID'
      ''',
      year, month, day
    ) AS subquery
    FROM all_dates
  )
);

EXECUTE IMMEDIATE FORMAT(
  ''' 
  SELECT
  lp.landmarkid,
  p.poitypeid,
    t.poiid, p.namelang1, 
    COUNT(1) AS cnt
  FROM (
    %s
  ) t
  LEFT JOIN `openrice-production.openrice3.poi` p
    ON t.poiid = p.poiid
  LEFT JOIN `openrice-production.openrice3.landmarkpoi` lp
    ON lp.poiid = p.poiid
  WHERE
    p.regionid = 0
    AND lp.landmarkid in (21,9313)
    AND p.status=10
  GROUP BY lp.landmarkid, p.poitypeid, t.poiid, p.namelang1
  ORDER BY lp.landmarkid, p.poitypeid, cnt desc
  ''',
  query
);
"""

In [ ]:
df_top5 = client.query(sql_3).to_dataframe()
df_top5

In [ ]:
# Banner View IMP (banner_view)

sql_4 = r"""
DECLARE sql STRING;

SET sql = (
  SELECT STRING_AGG(
    FORMAT('''
      SELECT
        event.item.Value AS value
      FROM `openrice-production.ORGA.PV_%s`,
      UNNEST(EventLabel.list) AS event
      WHERE EventAction = 'impression.ad.1'
        AND event.item.Param = 'FeatureItemID'
        AND SAFE_CAST(event.item.Value AS INT64) IN (42134,41971)
    ''', FORMAT_DATE('%Y%m%d', d))
  , ' UNION ALL ')
  FROM UNNEST(GENERATE_DATE_ARRAY('2026-09-01','2026-09-30')) d
);

EXECUTE IMMEDIATE
FORMAT('''
SELECT
  value,
  COUNT(1) AS cnt
FROM (
  %s
)
GROUP BY value
''', sql);

"""


In [ ]:
df_banner_view = client.query(sql_4).to_dataframe()
df_banner_view

In [ ]:
# Banner Click (banner_click)

sql_5 = r"""
DECLARE sql STRING;

SET sql = (
  SELECT STRING_AGG(
    FORMAT('''
      SELECT
        REGEXP_EXTRACT(event.item.Value, r'clk\\?id=(\\d+)') AS clk_id
      FROM `openrice-production.ORGA.PV_%s`,
      UNNEST(EventLabel.list) AS event
      WHERE EventAction = 'or.SR1.ad.1.URL'
        AND event.item.Param = 'url'
        AND REGEXP_CONTAINS(event.item.Value, r'clk\\?id=\\d+')
        AND SAFE_CAST(REGEXP_EXTRACT(event.item.Value, r'clk\\?id=(\\d+)') AS INT64) IN
        (7309685695,7289878736)
    ''', FORMAT_DATE('%Y%m%d', d))
  , ' UNION ALL ')
  FROM UNNEST(GENERATE_DATE_ARRAY('2026-09-01','2026-09-30')) d
);

EXECUTE IMMEDIATE
FORMAT('''
SELECT
  clk_id,
  COUNT(1) AS cnt
FROM (
  %s
)
GROUP BY clk_id
''', sql);

"""

In [ ]:
df_banner_click = client.query(sql_5).to_dataframe()
df_banner_click